In [120]:
# =========================================================
# MOUNT GOOGLE DRIVE
# =========================================================

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [121]:
# =========================================================
# IMPORT LIBRARIES
# =========================================================

import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)

print("Libraries Imported")

Libraries Imported


In [122]:
# =========================================================
# LOAD IMPUTED DATASETS
# =========================================================

train_path = '/content/drive/MyDrive/Traffic_Prediction/dataset/train_imputed.csv'
test_path = '/content/drive/MyDrive/Traffic_Prediction/dataset/test_imputed.csv'

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print("Train Shape :", train_df.shape)
print("Test Shape  :", test_df.shape)

Train Shape : (77299, 16)
Test Shape  : (41778, 16)


In [123]:
# =========================================================
# COMBINE TRAIN + TEST
# =========================================================

combined = pd.concat([train_df, test_df], axis=0)

print("Combined Shape :", combined.shape)

Combined Shape : (119077, 16)


In [124]:
# =========================================================
# INSPECT TIMESTAMP AND DAY
# =========================================================

print(train_df['timestamp'].head(20))

print("\n")

print(train_df['day'].head(20))

0     1970-02-18 00:00:00
1     1970-02-18 00:00:00
2     1970-02-18 00:00:00
3     1970-02-18 00:00:00
4     1970-02-18 00:00:00
5     1970-02-18 00:00:00
6     1970-02-18 00:00:00
7     1970-02-18 00:00:00
8     1970-02-18 00:00:00
9     1970-02-18 00:00:00
10    1970-02-18 00:00:00
11    1970-02-18 00:00:00
12    1970-02-18 00:00:00
13    1970-02-18 00:00:00
14    1970-02-18 00:00:00
15    1970-02-18 00:00:00
16    1970-02-18 00:00:00
17    1970-02-18 00:00:00
18    1970-02-18 00:00:00
19    1970-02-18 00:00:00
Name: timestamp, dtype: object


0     48
1     48
2     48
3     48
4     48
5     48
6     48
7     48
8     48
9     48
10    48
11    48
12    48
13    48
14    48
15    48
16    48
17    48
18    48
19    48
Name: day, dtype: int64


In [125]:
# =========================================================
# UNIQUE DATES
# =========================================================

temp_dates = pd.to_datetime(train_df['timestamp'])

print(temp_dates.dt.date.nunique())

2


In [126]:
# =========================================================
# TIMESTAMP ANALYSIS
# =========================================================

print(train_df['timestamp'].min())
print(train_df['timestamp'].max())

1970-02-18 00:00:00
1970-02-19 02:00:00


In [127]:
train_df['timestamp'] = pd.to_datetime(train_df['timestamp'])

print(train_df['timestamp'].dt.hour.value_counts().sort_index())

timestamp
0     6338
1     6871
2     4455
3     3595
4     3683
5     3621
6     3665
7     3569
8     3507
9     3549
10    3622
11    3567
12    3525
13    3271
14    3000
15    2580
16    2108
17    1684
18    1247
19    1214
20    1418
21    1738
22    2410
23    3062
Name: count, dtype: int64


In [128]:
# =========================================================
# BASIC TIME FEATURES
# =========================================================

combined['timestamp'] = pd.to_datetime(combined['timestamp'])
combined['hour'] = combined['timestamp'].dt.hour

print("Hour Feature Created")

Hour Feature Created


In [129]:
# =========================================================
# CYCLICAL HOUR ENCODING
# =========================================================

combined['hour_sin'] = np.sin(
    2 * np.pi * combined['hour'] / 24
)

combined['hour_cos'] = np.cos(
    2 * np.pi * combined['hour'] / 24
)

print("Cyclical Hour Encoding Done")

Cyclical Hour Encoding Done


In [130]:
# =========================================================
# PEAK HOUR FEATURE
# =========================================================

combined['is_peak_hour'] = combined['hour'].isin(
    [0,1,2,3,4,5,6,7,8,9,10,11,12]
).astype(int)

print("Peak Hour Feature Created")

Peak Hour Feature Created


In [131]:
# =========================================================
# DAY CYCLICAL ENCODING
# =========================================================

combined['day_sin'] = np.sin(
    2 * np.pi * combined['day'] / 7
)

combined['day_cos'] = np.cos(
    2 * np.pi * combined['day'] / 7
)

print("Day Cyclical Encoding Done")

Day Cyclical Encoding Done


In [132]:
# =========================================================
# DAY GROUPS
# =========================================================

combined['day_group'] = pd.cut(
    combined['day'],
    bins=5,
    labels=False
)

print("Day Groups Created")

Day Groups Created


In [133]:
# =========================================================
# GEOHASH FREQUENCY ENCODING
# =========================================================

geo_freq = combined['geohash'].value_counts()

combined['geohash_freq'] = combined['geohash'].map(geo_freq)

print("Geohash Frequency Encoding Done")

Geohash Frequency Encoding Done


In [134]:
# =========================================================
# GEOHASH PREFIX FEATURES
# =========================================================

combined['geohash_4'] = combined['geohash'].str[:4]

combined['geohash_5'] = combined['geohash'].str[:5]

print("Geohash Prefix Features Created")

Geohash Prefix Features Created


In [135]:
# =========================================================
# ROAD CAPACITY FEATURE
# =========================================================

combined['road_capacity'] = (
    combined['NumberofLanes']
)

print("Road Capacity Feature Created")

Road Capacity Feature Created


In [137]:
# =========================================================
# LARGE VEHICLE INTERACTION
# =========================================================

combined['LargeVehicles_encoded'] = (
    combined['LargeVehicles']
    .astype('category')
    .cat.codes
)

combined['heavy_vehicle_lane_interaction'] = (
    combined['NumberofLanes']
    *
    combined['LargeVehicles_encoded']
)

print("Heavy Vehicle Interaction Feature Created")

Heavy Vehicle Interaction Feature Created


In [138]:
# =========================================================
# LANDMARK INTERACTION FEATURE
# =========================================================

combined['Landmarks_encoded'] = (
    combined['Landmarks']
    .astype('category')
    .cat.codes
)

combined['landmark_lane_interaction'] = (
    combined['NumberofLanes']
    *
    combined['Landmarks_encoded']
)

print("Landmark Interaction Feature Created")

Landmark Interaction Feature Created


In [139]:
# =========================================================
# TEMPERATURE BINS
# =========================================================

combined['temp_bin'] = pd.cut(
    combined['Temperature'],
    bins=6,
    labels=False
)

print("Temperature Bins Created")

Temperature Bins Created


In [140]:
# =========================================================
# WEATHER + TEMPERATURE INTERACTION
# =========================================================

combined['weather_temp_interaction'] = (
    combined['Weather'].astype(str)
    + "_"
    + combined['temp_bin'].astype(str)
)

print("Weather-Temperature Interaction Created")

Weather-Temperature Interaction Created


In [141]:
# =========================================================
# WEATHER SEVERITY FEATURE
# =========================================================

weather_map = {
    'Sunny': 0,
    'Rainy': 1,
    'Foggy': 2,
    'Snowy': 3
}

combined['weather_severity'] = (
    combined['Weather']
    .map(weather_map)
)

print("Weather Severity Feature Created")

Weather Severity Feature Created


In [142]:
# =========================================================
# GEOHASH DEMAND MEAN
# =========================================================

geo_demand_mean = train_df.groupby(
    'geohash'
)['demand'].mean()

combined['geohash_demand_mean'] = (
    combined['geohash']
    .map(geo_demand_mean)
)

print("Geohash Demand Mean Feature Created")

Geohash Demand Mean Feature Created


In [144]:
# =========================================================
# HANDLE UNSEEN GEOHASHES
# =========================================================

global_demand_mean = train_df['demand'].mean()

combined['geohash_demand_mean'].fillna(
    global_demand_mean,
    inplace=True
)

print("Unseen geohashes handled")

Unseen geohashes handled


In [145]:
print(
    combined['geohash_demand_mean']
    .isnull()
    .sum()
)

0


In [148]:
# =========================================================
# HOUR DEMAND MEAN
# =========================================================

hour_demand_mean = train_df.groupby(
    'hour'
)['demand'].mean()

combined['hour_demand_mean'] = (
    combined['hour']
    .map(hour_demand_mean)
)

print("Hour Demand Mean Feature Created")

Hour Demand Mean Feature Created


In [149]:
# =========================================================
# DAY DEMAND MEAN
# =========================================================

day_demand_mean = train_df.groupby(
    'day'
)['demand'].mean()

combined['day_demand_mean'] = (
    combined['day']
    .map(day_demand_mean)
)

print("Day Demand Mean Feature Created")

Day Demand Mean Feature Created


In [151]:
# =========================================================
# ROADTYPE DEMAND MEAN
# =========================================================

road_demand_mean = train_df.groupby(
    'RoadType'
)['demand'].mean()

combined['roadtype_demand_mean'] = (
    combined['RoadType']
    .map(road_demand_mean)
)

print("RoadType Demand Mean Feature Created")

RoadType Demand Mean Feature Created


In [152]:
# =========================================================
# GEOHASH + HOUR DENSITY
# =========================================================

geo_hour_density = combined.groupby(
    ['geohash', 'hour']
).size()

combined['geo_hour_density'] = (
    combined.set_index(['geohash', 'hour'])
    .index
    .map(geo_hour_density)
)

print("Geo-Hour Density Feature Created")

Geo-Hour Density Feature Created


In [153]:
# =========================================================
# GEOHASH + DAY DENSITY
# =========================================================

geo_day_density = combined.groupby(
    ['geohash', 'day']
).size()

combined['geo_day_density'] = (
    combined.set_index(['geohash', 'day'])
    .index
    .map(geo_day_density)
)

print("Geo-Day Density Feature Created")

Geo-Day Density Feature Created


In [154]:
# =========================================================
# DROP RAW TIMESTAMP
# =========================================================

combined.drop(columns=['timestamp'], inplace=True)

print("Raw Timestamp Dropped")

Raw Timestamp Dropped


In [158]:
# =========================================================
# CHECK NULL VALUES
# =========================================================

combined.isnull().sum()

,0
Index,0
geohash,0
day,0
demand,41778
RoadType,0
NumberofLanes,0
LargeVehicles,0
Landmarks,0
Temperature,0
Weather,0


In [159]:
combined.drop(columns=['dayofweek'], inplace=True)

KeyError: "['dayofweek'] not found in axis"

In [160]:
# =========================================================
# SPLIT TRAIN AND TEST
# =========================================================

train_fe = combined.iloc[:len(train_df)].copy()

test_fe = combined.iloc[len(train_df):].copy()

print("Train Shape :", train_fe.shape)
print("Test Shape  :", test_fe.shape)

Train Shape : (77299, 37)
Test Shape  : (41778, 37)


In [161]:
test_fe.drop(columns=['demand'], inplace=True)

print("Demand column removed from test")

Demand column removed from test


In [162]:
print(train_fe.isnull().sum())

print("\n")

print(test_fe.isnull().sum())

print("\n")

print("Target Present in Train :", 'demand' in train_fe.columns)

print("Target Present in Test  :", 'demand' in test_fe.columns)

Index                             0
geohash                           0
day                               0
demand                            0
RoadType                          0
NumberofLanes                     0
LargeVehicles                     0
Landmarks                         0
Temperature                       0
Weather                           0
hour                              0
Temperature_missing               0
Weather_missing                   0
RoadType_missing                  0
hour_sin                          0
hour_cos                          0
is_peak_hour                      0
day_sin                           0
day_cos                           0
day_group                         0
geohash_freq                      0
geohash_4                         0
geohash_5                         0
road_capacity                     0
LargeVehicles_encoded             0
heavy_vehicle_lane_interaction    0
Landmarks_encoded                 0
landmark_lane_interaction   

In [165]:
# =========================================================
# VERIFY TARGET
# =========================================================

print("Target Present in Train :", 'demand' in train_fe.columns)

print("Target Present in Test  :", 'demand' in test_fe.columns)

Target Present in Train : True
Target Present in Test  : False


In [166]:
# =========================================================
# SAVE FEATURE ENGINEERED DATASETS
# =========================================================

train_save_path = '/content/drive/MyDrive/Traffic_Prediction/dataset/train_feature_engineered.csv'

test_save_path = '/content/drive/MyDrive/Traffic_Prediction/dataset/test_feature_engineered.csv'

train_fe.to_csv(train_save_path, index=False)

test_fe.to_csv(test_save_path, index=False)

print("Feature Engineered Datasets Saved Successfully")

Feature Engineered Datasets Saved Successfully


In [118]:
test_fe['demand'].isnull().sum()

np.int64(41778)